In [1]:
raw_users = [
("U001","Amit","28","Hyderabad","['AI','ML','Cloud']"),
("U002","Neha","Thirty","Delhi","AI,Testing"),
("U003","Ravi",None,"Bangalore",["Data","Spark"]),
("U004","Pooja","29","Mumbai",None),
("U005","", "31","Chennai","['DevOps']")
]

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("age", StringType(), True),
    StructField("city", StringType(), True),
    StructField("skills", StringType(), True)
])

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Initialize SparkSession if it doesn't already exist
spark = SparkSession.builder.appName("UserProcessing").getOrCreate()

def normalize_age(age):
    try:
        return int(age)
    except:
        return None

normalize_age_udf = udf(normalize_age, IntegerType())
users_df = spark.createDataFrame(raw_users, schema)
users_df = users_df.withColumn("age", normalize_age_udf("age"))

In [7]:
from pyspark.sql.functions import split, col, when
from pyspark.sql.types import ArrayType, StringType
import ast # Import ast module for safe evaluation

# Normalize skills into ArrayType
def normalize_skills(skills):
    if skills is None:
        return []
    if isinstance(skills, str):
        if skills.startswith('[') and skills.endswith(']') or (skills.startswith('(') and skills.endswith(')')):
            try:
                # Use ast.literal_eval for safe evaluation of string literals
                parsed_list = ast.literal_eval(skills)
                # Ensure all elements in the parsed list are strings and strip them
                return [str(s).strip() for s in parsed_list if str(s).strip()]
            except (ValueError, SyntaxError):
                # Fallback for malformed string literals (e.g., "[Data,Spark]" or "(Data,Spark)")
                # Remove brackets/parentheses and split by comma
                skills_stripped = skills.strip('[]()')
                return [s.strip() for s in skills_stripped.split(',') if s.strip()]
        else:
            # Comma-separated string (e.g., "AI,Testing")
            return [s.strip() for s in skills.split(',') if s.strip()]
    elif isinstance(skills, list):
        # Already a Python list, ensure elements are strings and not empty after stripping
        return [str(s).strip() for s in skills if str(s).strip()]
    return []


normalize_skills_udf = udf(normalize_skills, ArrayType(StringType()))


users_df = users_df.withColumn("skills", normalize_skills_udf("skills"))

In [9]:
users_df = users_df.withColumn("name", when(col("name") == "", "Unknown").otherwise(col("name")))


In [14]:
raw_courses = [
("C001","PySpark Mastery",("Data Engineering","Advanced"),"₹9999"),
("C002","AI for Testers",{"domain":"QA","level":"Beginner"},"8999"),
("C003","ML Foundations",("AI","Intermediate"),None),
("C004","Data Engineering Bootcamp",('Data','Advanced'),"₹14999")
]

In [11]:
from pyspark.sql.types import StructType, StructField, StringType, MapType

course_schema = StructType([
    StructField("course_id", StringType(), True),
    StructField("course_name", StringType(), True),
    StructField("domain_info", StructType([
        StructField("domain", StringType(), True),
        StructField("level", StringType(), True)
    ]), True),
    StructField("price", StringType(), True)
])

In [15]:
from pyspark.sql.functions import col

courses_df = spark.createDataFrame(raw_courses, course_schema)

# Extract domain and level
courses_df = courses_df.withColumn("domain", col("domain_info.domain")) \
                       .withColumn("level", col("domain_info.level")) \
                       .drop("domain_info")

In [16]:
from pyspark.sql.functions import regexp_replace

def normalize_price(price):
    if price:
        return int(regexp_replace(price, '[^0-9]', ''))
    return None

courses_df = courses_df.withColumn("price", udf(normalize_price, IntegerType())("price"))

In [17]:
courses_df = courses_df.fillna({"price": 0})

In [19]:
raw_enrollments = [
("U001","C001","2024-01-05"),
("U002","C002","05/01/2024"),
("U003","C001","2024/01/06"),
("U004","C003","invalid_date"),
("U001","C004","2024-01-10")
]
from pyspark.sql.functions import to_date

def normalize_date(date_str):
    try:
        return to_date(date_str, "yyyy-MM-dd")
    except:
        return None

enrollments_df = spark.createDataFrame(raw_enrollments, ["user_id", "course_id", "enrollment_date"])
enrollments_df = enrollments_df.withColumn("enrollment_date", normalize_date("enrollment_date"))


In [20]:
enrollments_df = enrollments_df.filter(col("enrollment_date").isNotNull())

In [21]:
enrollments_with_users_df = enrollments_df.join(users_df, on="user_id", how="left")
enrollments_with_courses_df = enrollments_with_users_df.join(courses_df, on="course_id", how="left")

In [23]:
from pyspark.sql.functions import broadcast


enrollments_with_courses_df = enrollments_df.join(broadcast(users_df), on="user_id")

In [24]:
enrollments_with_courses_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [user_id])
:- Filter isnotnull(enrollment_date#81)
:  +- Project [user_id#78, course_id#79, to_date(enrollment_date#80, Some(yyyy-MM-dd), Some(Etc/UTC), true) AS enrollment_date#81]
:     +- LogicalRDD [user_id#78, course_id#79, enrollment_date#80], false
+- ResolvedHint (strategy=broadcast)
   +- Project [user_id#0, CASE WHEN (name#30 = ) THEN Unknown ELSE name#30 END AS name#49, age#6, city#3, skills#29]
      +- Project [user_id#0, CASE WHEN (name#9 = ) THEN Unknown ELSE name#9 END AS name#30, age#6, city#3, skills#29]
         +- Project [user_id#0, name#9, age#6, city#3, normalize_skills(skills#8)#28 AS skills#29]
            +- Project [user_id#0, CASE WHEN (name#1 = ) THEN Unknown ELSE name#1 END AS name#9, age#6, city#3, skills#8]
               +- Project [user_id#0, name#1, age#6, city#3, normalize_skills(skills#4)#7 AS skills#8]
                  +- Project [user_id#0, name#1, normalize_age(age#2)#5 AS age#6, city#3, skills#4]

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col, udf
from pyspark.sql.types import ArrayType, StringType

# Initialize SparkSession if it doesn't already exist
spark = SparkSession.builder.appName("UserProcessing").getOrCreate()

raw_activity = [
("U001","login,watch,logout","{'device':'mobile','ip':'1.1.1.1'}",120),
("U002","login,watch","device=laptop;ip=2.2.2.2",90), # Changed list to string
("U003","login|logout",None,30),
("U004",None,"{'device':'tablet'}",60)
]

def normalize_actions(actions):
    if isinstance(actions, str):
        return actions.split(",")  # Split comma-separated actions
    elif isinstance(actions, list):
        return actions
    return []

activity_df = spark.createDataFrame(raw_activity, ["user_id", "actions", "metadata", "duration"])
activity_df = activity_df.withColumn("actions", udf(normalize_actions, ArrayType(StringType()))("actions"))

In [5]:
from pyspark.sql.functions import from_json, udf
from pyspark.sql.types import MapType, StringType # Import MapType and StringType
import json # Import json module

def normalize_metadata(metadata):
    if metadata:
        return json.loads(metadata)
    return {}

activity_df = activity_df.withColumn("metadata", udf(normalize_metadata, MapType(StringType(), StringType()))("metadata"))

In [7]:
from pyspark.sql.functions import when, col, array

activity_df = activity_df.withColumn("actions", when(col("actions").isNull(), array()).otherwise(col("actions")))

In [8]:
from pyspark.sql.functions import explode

activity_exploded_df = activity_df.withColumn("action", explode(col("actions"))).groupBy("action").count()

In [9]:
activity_exploded_df.show()

+------------+-----+
|      action|count|
+------------+-----+
|       watch|    2|
|      logout|    1|
|       login|    2|
|login|logout|    1|
+------------+-----+



In [11]:
raw_payments = [
("U001","2024-01-05",9999),
("U001","2024-01-10",14999),
("U002","2024-01-06",8999),
("U003","2024-01-07",0),
("U004","2024-01-08",7999),
("U001","2024-01-15",1999)
]
payments_df = spark.createDataFrame(raw_payments, ["user_id", "payment_date", "amount"])

from pyspark.sql.functions import to_date, col
# Convert payment_date to standard date format
payments_df = payments_df.withColumn("payment_date", to_date(col("payment_date"), "yyyy-MM-dd"))

In [12]:
total_spend_df = payments_df.groupBy("user_id").sum("amount").withColumnRenamed("sum(amount)", "total_spend")

In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum # Import PySpark's sum function

window_spec = Window.partitionBy("user_id").orderBy("payment_date")
payments_df = payments_df.withColumn("running_total", sum("amount").over(window_spec))

In [15]:
from pyspark.sql.functions import rank

ranked_df = total_spend_df.withColumn("rank", rank().over(Window.orderBy(col("total_spend").desc())))

In [16]:
total_spend_df.show()
payments_df.show()

+-------+-----------+
|user_id|total_spend|
+-------+-----------+
|   U002|       8999|
|   U001|      26997|
|   U004|       7999|
|   U003|          0|
+-------+-----------+

+-------+------------+------+-------------+
|user_id|payment_date|amount|running_total|
+-------+------------+------+-------------+
|   U001|  2024-01-05|  9999|         9999|
|   U001|  2024-01-10| 14999|        24998|
|   U001|  2024-01-15|  1999|        26997|
|   U002|  2024-01-06|  8999|         8999|
|   U003|  2024-01-07|     0|            0|
|   U004|  2024-01-08|  7999|         7999|
+-------+------------+------+-------------+

